## Lab-10 Object Detection using Neural Network Accelerator
*Suggested time: 45 minutes*

Hardware: **GPU**

This lab demonstrates model preparation for NN Accelerator. It comprises of the following steps.
- Get images for classification
- Obtain trained CNN model for 11 types of sound
- Convert and Quantize trained model
- Compare accuracy of floating point and quantized model
- Compile quantized model for Accelerator

The compiled model will be downloaded to the Accelerator for demonstration by the instructor.

**Part B:**

Execute ResNet50 ONNX model using ONNX runtime and GPU

This lab is based on the following Notebook on [Retrain a classification model for Edge TPU using post-training quantization (with TF2)](https://colab.research.google.com/github/google-coral/tutorials/blob/master/retrain_classification_ptq_tf2.ipynb#scrollTo=TaX0smDP7xQY)

#### NN Accelerator CNN model for ESC-11 dataset

In this lab, we train a CNN model using spectrograms of audio from the dataset.  We convert it to TensorFlow Lite using post-training quantization. Finally, we compile it for compatibility with the Edge TPU (available in [Coral devices](https://gweb-coral-full.uc.r.appspot.com/)).


##### **Step 1:**

Import required libraries. . 

In [0]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt

##### **Step 2:**

Use ESC-11 dataset to train CNN model. The following code randomizes and divides up the spectrograms into training and validation sets, and generates a labels file based on the sound folder names. The next cell sets up the sound directory which contains spectrograms from different audio sources.

In [0]:
!wget -q https://edge-ai-doulos.s3.us-west-2.amazonaws.com/esc_spec.zip
!unzip -q esc_spec

In [0]:
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'esc_spec')
LABEL_FILE = os.path.join(BASE_DIR, 'sound_labels.txt')
SOUND_DIR = os.path.join(BASE_DIR, 'esc_spec')


##### **Step 3:**

As before, we use [`ImageDataGenerator`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) to rescale the image data into float values (divide by 255 so the tensor values are between 0 and 1). It is here that we can specify the training/validation split ratio. Keras  API call `flow_from_directory()` creates two generators: one for the training dataset and one for the validation dataset.


In [0]:
IMAGE_SIZE = 224; BATCH_SIZE = 11

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='training')

val_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation')

##### **Step 4:**

On each iteration, these generators provide a batch of images by reading images from different folders on disk and processing them to the proper tensor size (224 x 224). The output is a tuple of (images, labels). The shapes of image and label batch are accessed for confirmation.

In [0]:
image_batch, label_batch = next(val_generator)
image_batch.shape, label_batch.shape

Save class labels to a text file:

In [0]:
print (train_generator.class_indices)

labels = '\n'.join(sorted(train_generator.class_indices.keys()))

with open(LABEL_FILE, 'w') as f:
  f.write(labels)

###### **Step 5:**

Create a fully convolutional neural network (FCNN) .  Instead of using only fully connected(dense layer) as classifer, the FCNN uses a combination of Global Average Pooling and fully connected layer as a classifier.


In [0]:
import warnings
warnings.filterwarnings("ignore")

from keras import layers
from keras import models

model = models.Sequential()
model.add(layers.InputLayer(shape=(224, 224, 3)))
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(11, activation='softmax'))

##### **Step 6:**

Confirm the model's architecture using `summary()` method, and compare the trainable parameters to the LeNet model created in lab 4.  Why do you think this model has less trainable parameters?

The summary of LeNet from lab 4 is given below:
- Total params: 9,684,171 (36.94 MB)
- Trainable params: 9,684,171 (36.94 MB)
- Non-trainable params: 0 (0.00 B)



In [0]:
model.summary()

This model has less trainable parameters due to the use of Global Average Pooling in the classifier section of the model.  The LeNet model in lab 4 used multiple dense layers for classification purposes.

##### **Step 7:**

Now we train the model using data provided by the `train_generator` and `val_generator` that we created at the beginning. The history object generated using model training is saved as a Python pickle object. The trained model and the history object are fetched below.


In [0]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])



history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=50,
                    validation_data=val_generator,
                    validation_steps=len(val_generator))

##### **Step 8:**

Study the training and validation accuracy curves obtained using the history object saved from the output of model.fit().


In [0]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.xlabel('epoch')
plt.show()

##### Question: 
Looking at the training and validation curves, do you think the model is overfitting or underfitting?

#### Solution

<details>

<summary> Our answer </summary>

The model seems to be overfitting, as it is performing very well on the training data and not as well on the unseen validation data. It is not generalizing.

</details>



##### **Step: 9**

Convert trained CNN model to TFLite Flatbuffer format. We start by creating a basic (un-quantized) TensorFlow Lite model.

In [0]:
CNN_TFLITE_MODEL = os.path.join(BASE_DIR, 'simple_cnn.tflite')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(CNN_TFLITE_MODEL, 'wb') as f:
  f.write(tflite_model)

##### **Step 10:**

This `.tflite` file still uses floating-point values for the parameter data. For the model to work on the NN Accelerator we need to fully quantize the model to int8 format.

To fully quantize the model, we need to perform post-training quantization (which retains model accuracy).  Post-training quantization starts with  a representative dataset, which evaluates the dataset to determine quantization value range.

In [0]:
CNN_QUAN_TFLITE_MODEL = os.path.join(BASE_DIR, 'simple_cnn_quant.tflite')

# A generator that provides a representative dataset
def representative_data_gen():
  dataset_list = tf.data.Dataset.list_files(SOUND_DIR + '/*/*')
  for i in range(100):
    image = next(iter(dataset_list))
    image = tf.io.read_file(image)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image / 255., tf.float32)
    image = tf.expand_dims(image, 0)
    yield [image]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
# This enables quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# This sets the representative dataset for quantization
converter.representative_dataset = representative_data_gen
# This ensures that if any ops can't be quantized, the converter throws an error
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# For full integer quantization, though supported types defaults to int8 only, we explicitly declare it for clarity.
converter.target_spec.supported_types = [tf.int8]
# These set the input and output tensors to uint8 (added in r2.3)
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()

with open(CNN_QUAN_TFLITE_MODEL, 'wb') as f:
  f.write(tflite_model)

##### **Step 11:**

Compare accuracy of 32-bit floating-point unquantized TFLite model with int8 quantized TFLite model.
Since there is no API to evaluate the accuracy of a TensorFlow Lite model, we use the TFLite (LiteRT) runtime. This code runs several inferences and compares the predictions against ground truth.

In [0]:
!pip install ai_edge_litert

In [0]:
from ai_edge_litert.interpreter import Interpreter

def set_input_tensor(interpreter, input):
  input_details = interpreter.get_input_details()[0]
  tensor_index = input_details['index']
  input_tensor = interpreter.tensor(tensor_index)()[0]
  input_tensor[:, :] = input

def classify_image(interpreter, input):
  set_input_tensor(interpreter, input)
  interpreter.invoke()
  output_details = interpreter.get_output_details()[0]
  output = interpreter.get_tensor(output_details['index'])
  top_1 = np.argmax(output)
  return top_1

# Recreate the validation generator for the TFLite unquantized model
val_generator_tflite_unquant = datagen.flow_from_directory(
    'esc_spec',
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    shuffle=False) # Ensure consistent order

interpreter = Interpreter('simple_cnn.tflite')
interpreter.allocate_tensors()

# Collect all inference predictions in a list
all_predictions_tflite_unquant = []
all_truths_tflite_unquant = []

for _ in range(len(val_generator_tflite_unquant)): # Iterate for all batches
  image_batch_tflite_unquant, label_batch_tflite_unquant = next(val_generator_tflite_unquant)
  batch_truth_tflite_unquant = np.argmax(label_batch_tflite_unquant, axis=1)

  for j in range(len(image_batch_tflite_unquant)):
    # classify_image expects a single image, so expand_dims is needed
    prediction = classify_image(interpreter, np.expand_dims(image_batch_tflite_unquant[j], axis=0))
    all_predictions_tflite_unquant.append(prediction)
    all_truths_tflite_unquant.append(batch_truth_tflite_unquant[j])

# Compare all predictions to the ground truth
tflite_unquant_accuracy = tf.keras.metrics.Accuracy()
tflite_unquant_accuracy(all_predictions_tflite_unquant, all_truths_tflite_unquant)
print("TF Lite unquantized accuracy: {:.3%}".format(tflite_unquant_accuracy.result()))

In [0]:
from ai_edge_litert.interpreter import Interpreter

def set_input_tensor(interpreter, input):
  input_details = interpreter.get_input_details()[0]
  tensor_index = input_details['index']
  input_tensor = interpreter.tensor(tensor_index)()[0]
  # Inputs for the TFLite model must be uint8, so we quantize our input data.
  # NOTE: This step is necessary only because we're receiving input data from
  # ImageDataGenerator, which rescaled all image data to float [0,1]. When using
  # bitmap inputs, they're already uint8 [0,255] so this can be replaced with:
  #   input_tensor[:, :] = input
  scale, zero_point = input_details['quantization']
  input_tensor[:, :] = np.uint8(input / scale + zero_point)

def classify_image(interpreter, input):
  set_input_tensor(interpreter, input)
  interpreter.invoke()
  output_details = interpreter.get_output_details()[0]
  output = interpreter.get_tensor(output_details['index'])
  # Outputs from the TFLite model are uint8, so we dequantize the results:
  scale, zero_point = output_details['quantization']
  output = scale * (output - zero_point)
  top_1 = np.argmax(output)
  return top_1

# Recreate the validation generator for the TFLite quantized model
val_generator_tflite_quant = datagen.flow_from_directory(
    'esc_spec',
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    shuffle=False) # Ensure consistent order

interpreter = Interpreter('simple_cnn_quant.tflite')
interpreter.allocate_tensors()

# Collect all inference predictions in a list
all_predictions_tflite_quant = []
all_truths_tflite_quant = []

for _ in range(len(val_generator_tflite_quant)): # Iterate for all batches
  image_batch_tflite_quant, label_batch_tflite_quant = next(val_generator_tflite_quant)
  batch_truth_tflite_quant = np.argmax(label_batch_tflite_quant, axis=1)

  for j in range(len(image_batch_tflite_quant)):
    # classify_image expects a single image, so expand_dims is needed
    prediction = classify_image(interpreter, np.expand_dims(image_batch_tflite_quant[j], axis=0))
    all_predictions_tflite_quant.append(prediction)
    all_truths_tflite_quant.append(batch_truth_tflite_quant[j])

# Compare all predictions to the ground truth
tflite_quant_accuracy = tf.keras.metrics.Accuracy()
tflite_quant_accuracy(all_predictions_tflite_quant, all_truths_tflite_quant)
print("Quant TF Lite accuracy: {:.3%}".format(tflite_quant_accuracy.result()))


##### **Observation:** 

You might see some, but hopefully not very much accuracy drop between the raw model and the TensorFlow Lite model.

For example When running step 12, the following results were found:
- Raw model accuracy: 63.636%
- Quant TF Lite accuracy: 81.818%

Are these results expected? Is it a bug in the code evaluating the model performance? If not, how can it be that the quantized model performs much better than the raw model?

#### Solution

<details>

<summary> Our answer </summary>
This was a lucky strike, where a batch chosen just happens to perform better with the quantized model than the raw model. Re-running the model’s performance evaluation on other batches lead more consistent result.

- Raw model accuracy: 72.727%
- Quant TF Lite accuracy: 54.545%
- Raw model accuracy: 90.909%
- Quant TF Lite accuracy: 90.909%
- Raw model accuracy: 63.636%
- Quant TF Lite accuracy: 63.636%
- Raw model accuracy: 72.727%
- Quant TF Lite accuracy: 72.727%
</details>







##### **Step 12:**

Compile quantized (int8) for the [Coral NN Accelerator](https://gweb-coral-full.uc.r.appspot.com/)(Edge TPU) device. First, we need to install the Edge TPU compiler to compile our model.


In [0]:
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get -q update
!sudo apt-get install -q edgetpu-compiler

**Step 12(a):**

Run model compiler with saved quantized model file name as argument.

In [0]:
!edgetpu_compiler simple_cnn_quant.tflite

**Step 12(b):**

The compiled model generated has the same filename with "_edgetpu" appended at the end. Take a look at the log file for determining if any operators have been delegated to the CPU.

In [0]:
!cat simple_cnn_quant_edgetpu.log

#### Questions:

A. How would you know what operators are not supported by the Coral Accelerator?

B. What happens if the model contains an operator not supported by the Accelerator?




#### Solution

<details>

<summary> Our answer </summary>
    
 -  You will have to refer to the documentation located on Coral [website](https://coral.ai/docs/edgetpu/models-intro/#supported-operations)
    
 -  The unsupported operator is sent to the CPU for execution. This causes model execution to slow down. It is best to be mindful of the supported operators and use them, while creating the model.

</details>
 

##### **Step 13:**

Save the quantized and compiled model to your local computer.  This is done by downloading the model from the **Output** tab on the right panel. 

##### **Step 14:**

Execute the ESC-11 model on the Edge TPU,

You can now run the model on your Coral device with acceleration on the Edge TPU.

To get started, try using your `.tflite` model with [this code for image classification with the TensorFlow Lite API](https://github.com/google-coral/tflite/tree/master/python/examples/classification).

Just follow the instructions on that page to set up your device, copy the `simple_cnn_quant_edgetpu.tflite` and `sound_labels.txt` files to your Coral Dev Board or device with a Coral Accelerator, and pass it a spectrogram png like this:

```
python3 classify_image.py \
  --model simple_cnn_quant_edgetpu.tflite \
  --labels sound_labels.txt \
  --input spectrogram.png
```

Check out more examples for running inference at [coral.ai/examples](https://coral.ai/examples/#code-examples/).




#### **Part B: Inferencing using GPU:**

Use of ONNX runtime to execute model on GPU



##### **Step 0:**

- Download Imagenet classes file, sample image file and ResNet50 ONNX model

In [0]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/ONNX-ResNet.zip
!unzip ONNX-ResNet.zip

##### **Step 1:**

- Change runtime to T4-GPU
- Install onnxruntime-gpu package

In [0]:
!pip install -q onnxruntime-gpu

##### **Step 2:**

- Check if GPU device is available
- Experiment with lines 14 and 15 that define the execution provider to ONNX runtime
- Uncomment line 14 and comment line 15 if you wish to use GPU CUDA based execution.
- Comment line 14 and uncomment line 15 if you only wish to use just the CPU for execution
- Compare the reported execution time for both GPU(CUDA) and CPU options.
- *Run the cell multiple times (for both GPU and CPU) to determine the execution time*
- *The first run usually takes more time.*


In [0]:
import os
import onnxruntime as rt
import numpy as np
import time
import cv2 # Import OpenCV

BASE_DIR = os.getcwd()
ONNX_RESNET_MODEL = os.path.join(BASE_DIR, 'model.onnx')
DOG_IMAGE = os.path.join(BASE_DIR, 'dog.jpg')
LABELS_FILE = os.path.join(BASE_DIR, 'imagenet_classes.txt')


# 1. Verify GPU availability
print(f"ONNX Runtime device: {rt.get_device()}") # Should output 'GPU' if installed correctly

# 2. Define the path to your ResNet-50 ONNX model
# Set using ONNX_RESNET_MODEL

# 3. Create an inference session, prioritizing the CUDA Execution Provider
# First obtain a list of execution providers. The T4 GPU is a CUDA accelerator
# Subsequently experiment with different execution providers and see changes in model inference time

print(f"Available ONNX execution providers: {rt.get_available_providers()}")

providers = ['CUDAExecutionProvider']
#providers = ['CPUExecutionProvider']
sess = rt.InferenceSession(ONNX_RESNET_MODEL, providers=providers)

# Get input name and shape
input_name = sess.get_inputs()[0].name
input_shape = sess.get_inputs()[0].shape
print(f"Input name: {input_name}, Input shape: {input_shape}")

# 4. Load and preprocess an image (e.g., 'dog.jpg')
# Assuming 'dog.jpg' and 'imagenet_classes.txt' are available in the environment.
image_path = DOG_IMAGE # Replace with your image path
try:
    # Use OpenCV to read image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image file '{image_path}' could not be read. Check path or file integrity.")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
except FileNotFoundError:
    print(f"Error: Image file '{image_path}' not found. Please ensure it's downloaded.")
    raise

# ResNet-50 preprocessing: resize and normalize
img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

# Convert to numpy array and normalize
input_image = np.array(img).astype(np.float32)
input_image = input_image / 255.0 # Scale to [0, 1]

# ImageNet normalization parameters
# Explicitly cast mean and std to float32
mean = np.array([0.485, 0.456, 0.406]).astype(np.float32)
std = np.array([0.229, 0.224, 0.225]).astype(np.float32)
input_image = (input_image - mean) / std

# Transpose from HWC to CHW and add batch dimension
input_image = input_image.transpose(2, 0, 1) # C, H, W
input_image = np.expand_dims(input_image, axis=0) # Add batch dimension: 1, C, H, W

# 5. Run inference and measure time
start_time = time.time()
outputs = sess.run(None, {input_name: input_image})
end_time = time.time()

print(f"Inference time: {end_time - start_time:.4f} seconds")

# 6. Post-process the output
# Get the class ID with the highest probability
class_id = np.argmax(outputs[0])

# Load ImageNet labels
labels_file_path = LABELS_FILE
try:
    with open(labels_file_path, "r") as f:
        imagenet_labels = [line.strip() for line in f.readlines()]
except FileNotFoundError:
    print(f"Error: Labels file '{labels_file_path}' not found. Please ensure it's downloaded.")
    raise

# Get the label
label = imagenet_labels[class_id]

print(f"Predicted class ID: {class_id}")
print(f"Predicted label: {label}")

##### **Discussion:**

- Did you see a speedup when executing classification model inference of a single image on GPU?
- What would you do to see a meaningful speedup?

##### Solution

<details>
<summary> Our answer </summary>
- Use a more complex task such as object detection
- Try to work with batch of images instead of a single image
</details>





**Demo and Exercise:**

Examples such as object classification and object detection using Accelerator hardware will be demonstrated by your instructor using a Linux SBC.

- Train a custom classification model (using step enumerated above), quantize it and compile it for accelerator execution. Check to see if all operators can run on the accelerator. If the operator is not supported by the accelerator, it would be mentioned in the log file.  

- In case you have access to an accelerator module, run the model and determine speed up vis-a-vis CPU.